# Goal 2 · IAM ownership boundary

**Goal:** close the over-broad IAM policy so the vector agent can no longer read another agent's private session memory.

**Where:** IAM (console or CLI). There's no code change and **no evidence upload**: this goal is checked against IAM itself.

### Concepts

- **Ownership.** Each agent owns its data. Agent A (Intake) owns the conversational memory in `claims-sessions`. The vector agent (`McpVectorRole`) owns OpenSearch access and has no business reading sessions.
- **Identity-based policies are unioned.** If **any** attached or inline policy allows an action, it's allowed unless an explicit `Deny` matches. One stray policy is enough to break the boundary.
- **The role's two inline policies:**
  - `vector-store-access`: OpenSearch access plus the lab's deny rules. This one is legitimate; keep it.
  - `vector-memory-leak`: statement `VectorStoreAccess` allows `dynamodb:GetItem` and `dynamodb:Query` on `claims-sessions`. The name is misleading, and that's the leak.

In [ ]:
import os, sys, json, time
sys.path.insert(0, os.path.abspath('.'))
import labkit, boto3

ROLE = os.environ['MCP_VECTOR_ROLE_NAME']
TABLE_ARN = os.environ['SESSION_TABLE_ARN']
iam = boto3.client('iam', region_name=labkit.REGION)
role_arn = iam.get_role(RoleName=ROLE)['Role']['Arn']
print('role:', role_arn)
for name in iam.list_role_policies(RoleName=ROLE)['PolicyNames']:
    doc = iam.get_role_policy(RoleName=ROLE, PolicyName=name)['PolicyDocument']
    print(f'\n--- inline policy {name}')
    print(json.dumps(doc, indent=2))
print('\nmanaged policies:', [p['PolicyName'] for p in iam.list_attached_role_policies(RoleName=ROLE)['AttachedPolicies']])

### Observe: two ways to prove the leak

1. **The policy simulator**, which is what the console's *Simulate* button calls. It shows the decision without making any request.
2. **Live**: assume the role and read Agent A's session. The lab's trust policy lets your account assume it, so you can test as the agent.

In [ ]:
def simulate():
    r = iam.simulate_principal_policy(PolicySourceArn=role_arn,
                                      ActionNames=['dynamodb:GetItem', 'dynamodb:Query', 'dynamodb:PutItem'],
                                      ResourceArns=[TABLE_ARN])
    for x in r['EvaluationResults']:
        stmt = [m['SourcePolicyId'] for m in x.get('MatchedStatements', [])]
        print(f"{x['EvalActionName']:18} {x['EvalDecision']:14} matched: {stmt}")


def read_as_vector_agent():
    c = boto3.client('sts').assume_role(RoleArn=role_arn, RoleSessionName='vector-agent-probe')['Credentials']
    ddb = boto3.client('dynamodb', region_name=labkit.REGION, aws_access_key_id=c['AccessKeyId'],
                       aws_secret_access_key=c['SecretAccessKey'], aws_session_token=c['SessionToken'])
    try:
        items = ddb.query(TableName=os.environ['SESSION_TABLE'], KeyConditionExpression='session_id = :s',
                          ExpressionAttributeValues={':s': {'S': 's-demo'}})['Items']
        print(f'LEAK: the vector agent read {len(items)} private turns, e.g.:', items[0]['content']['S'][:60] if items else '-')
    except Exception as e:
        print('blocked:', type(e).__name__, str(e)[:160])

simulate()
read_as_vector_agent()

`PutItem` is already denied by the lab's deny rules in `vector-store-access`. Reads aren't denied, so the leak policy's `Allow` wins.

## Task 2 · Remove the over-broad grant

The fix is to **delete the whole inline policy `vector-memory-leak`**:
- **Don't edit it.** An emptied or narrowed copy is still a policy nobody owns.
- **Don't touch `vector-store-access`.**

**Console:** IAM → Roles → search for `McpVectorRole` → *Permissions* tab → select `vector-memory-leak` → **Remove** → confirm.

**CLI** (equivalent):
```bash
aws iam delete-role-policy --role-name <MCP_VECTOR_ROLE_NAME> --policy-name vector-memory-leak
```

<details><summary>Hint 1</summary>Find the statement that names the <code>claims-sessions</code> table. Which policy is it in?</details>
<details><summary>Hint 2</summary>Why not add a <code>Deny</code> instead? It would work, but it leaves a dangling allow that a later edit can re-expose. Delete the grant at its source.</details>

If you'd rather do it from here, set `CONFIRM = True` in the next cell.

In [ ]:
CONFIRM = False
if CONFIRM:
    iam.delete_role_policy(RoleName=ROLE, PolicyName='vector-memory-leak')
    print('deleted vector-memory-leak')
print('inline policies now:', iam.list_role_policies(RoleName=ROLE)['PolicyNames'])

## Verify

`GetItem` and `Query` against the `claims-sessions` ARN must now be **`implicitDeny`**: no policy allows them any more. IAM changes take a few seconds to propagate, so the live check takes a fresh session and retries briefly.

In [ ]:
simulate()
for attempt in range(3):
    read_as_vector_agent()
    time.sleep(5)
labkit.grade(2)

### Debrief

- Least privilege is enforced per principal, and **names lie**. Review what a policy *grants* (actions × resources), not what it's called.
- To keep the boundary from coming back, add a guardrail outside the role:
  - a permissions boundary on agent roles, or
  - an SCP or resource policy that only allows Agent A's role on `claims-sessions`.
- `terraform apply` in `infra/terraform` re-creates `vector-memory-leak`. Use it only if you want to reset this goal.